# Notebook for plotting a clustering
Needs to be run after `..._clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path(os.getcwd()), *Path(os.getcwd()).parents] if path.stem == "esa_consulting")
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
import plotnine as p9
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import TSNE
from umap import UMAP

from utils.plotting_utils import make_plot_2d_data
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_imputed = DatasetBundle.load(Path("data/imputed/data_discov_imputed"))
data_discov_imputed.meta = cast(pd.DataFrame, data_discov_imputed.meta)
data_discov_imputed.na = cast(pd.DataFrame, data_discov_imputed.na)
data_valid_imputed = DatasetBundle.load(Path("data/imputed/data_valid_imputed"))
data_valid_imputed.meta = cast(pd.DataFrame, data_valid_imputed.meta)
data_valid_imputed.na = cast(pd.DataFrame, data_valid_imputed.na)

data_discov_mfa = pd.read_parquet(Path("data/dim_red/data_discov_mfa.parquet"))
data_valid_mfa = pd.read_parquet(Path("data/dim_red/data_valid_mfa.parquet"))

proba_and_labels_discov = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_discov.parquet"))
proba_and_labels_valid = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_valid.parquet"))

# Set outputs folder
outputs_folder = "outputs/gmm_clustering"

# Set predicted probability and labels
proba_discov = proba_and_labels_discov.drop("label", axis="columns")
proba_valid = proba_and_labels_valid.drop("label", axis="columns")
labels_discov = proba_and_labels_discov["label"]
labels_valid = proba_and_labels_valid["label"]

In [ ]:
# Plot clustering in MFA dimensions
# comps_plots = (0, 1)
comps_plots = [(0, 1), (0, 2), (0, 3), (1, 2), (0, 4), (1, 3), (0, 5), (1, 4), (2, 3)]
qs = (0.005, 0.995)

# Plot clusterings
clusters_discov_mfa_plot = make_plot_2d_data(data_discov_mfa, comps=comps_plots, labels=labels_discov, quantiles=qs)
clusters_valid_mfa_plot = make_plot_2d_data(data_valid_mfa, comps=comps_plots, labels=labels_valid, quantiles=qs)

# Combine the plots
clusters_all_mfa_plot = (
    (clusters_discov_mfa_plot + p9.labs(title="Discovery clustering") + p9.theme(legend_position="none"))
    + (clusters_valid_mfa_plot + p9.labs(title="Validation clustering"))
) & p9.theme(figure_size=(10, 5))
clusters_all_mfa_plot.show()

# Save plots
# save_plot(clusters_discov_mfa_plot, Path(outputs_folder, "clusters_discov_plot_mfa.png"), dpi=150)
# save_plot(clusters_valid_mfa_plot, Path(outputs_folder, "clusters_valid_plot_mfa.png"), dpi=150)

save_plot(clusters_all_mfa_plot, Path(outputs_folder, "clusters_all_mfa_plot.png"), dpi=150)

In [ ]:
# Plot clustering using discrimination on labels
comps_plots = (0, 1)
# comps_plots = [(0, 1), (0, 2), (0, 3), (1, 2), (0, 4), (1, 3), (0, 5), (1, 4), (2, 3)]
qs = (0.005, 0.995)

# Fit discrimination model on discovery data and predicted labels
rng = np.random.default_rng(2395)
discrimination_model = LDA().fit(data_discov_mfa, labels_discov)

# Reduce data dimensionality using LDA
data_discov_lda = pd.DataFrame(discrimination_model.transform(data_discov_mfa)).rename(columns=lambda x: f"comp {x}")
data_valid_lda = pd.DataFrame(discrimination_model.transform(data_valid_mfa)).rename(columns=lambda x: f"comp {x}")

# Plot clusterings
clusters_discov_lda_plot = make_plot_2d_data(data_discov_lda, comps=comps_plots, labels=labels_discov, quantiles=qs)
clusters_valid_lda_plot = make_plot_2d_data(data_valid_lda, comps=comps_plots, labels=labels_valid, quantiles=qs)

# Combine the plots
clusters_all_lda_plot = (
    (clusters_discov_lda_plot + p9.labs(title="Discovery clustering") + p9.theme(legend_position="none"))
    + (clusters_valid_lda_plot + p9.labs(title="Validation clustering"))
) & p9.theme(figure_size=(10, 5))
clusters_all_lda_plot.show()

# Save plots
# save_plot(clusters_discov_lda_plot, Path(outputs_folder, "clusters_discov_plot_lda.png"), dpi=150)
# save_plot(clusters_valid_lda_plot, Path(outputs_folder, "clusters_valid_plot_lda.png"), dpi=150)

save_plot(clusters_all_lda_plot, Path(outputs_folder, "clusters_all_lda_plot.png"), dpi=150)

In [ ]:
# Plot projected clustering in 2d
comps_plots = (0, 1)
qs = (0.005, 0.995)
n_components = 2

# Define projection model
rng0 = np.random.default_rng(2395)
rng1, *_ = rng.spawn(1)
projection_method = "tsne"
proj_model_discov = TSNE(n_components=n_components, n_jobs=-1, random_state=np.random.RandomState(rng0.bit_generator))
proj_model_valid = TSNE(n_components=n_components, n_jobs=-1, random_state=np.random.RandomState(rng1.bit_generator))
# proj_model_discov = UMAP(n_components=n_components, n_jobs=1, random_state=np.random.RandomState(rng0.bit_generator))
# proj_model_valid = UMAP(n_components=n_components, n_jobs=1, random_state=np.random.RandomState(rng1.bit_generator))

# Reduce data dimensionality using the specified projection method
data_discov_proj = pd.DataFrame(proj_model_discov.fit_transform(data_discov_mfa)).rename(columns=lambda x: f"comp {x}")
data_valid_proj = pd.DataFrame(proj_model_valid.fit_transform(data_valid_mfa)).rename(columns=lambda x: f"comp {x}")

# Plot clusterings
clusters_discov_proj_plot = make_plot_2d_data(data_discov_proj, comps=comps_plots, labels=labels_discov, quantiles=qs)
clusters_valid_proj_plot = make_plot_2d_data(data_valid_proj, comps=comps_plots, labels=labels_valid, quantiles=qs)

# Combine the plots
clusters_all_proj_plot = (
    (clusters_discov_proj_plot + p9.labs(title="Discovery clustering") + p9.theme(legend_position="none"))
    + (clusters_valid_proj_plot + p9.labs(title="Validation clustering"))
) & p9.theme(figure_size=(10, 5))
clusters_all_proj_plot.show()

# Save plots
# save_plot(
#     clusters_discov_proj_plot, Path(outputs_folder, f"clusters_discov_plot_{projection_method.lower()}.png"), dpi=150
# )
# save_plot(
#     clusters_valid_proj_plot, Path(outputs_folder, f"clusters_valid_plot_{projection_method.lower()}.png"), dpi=150
# )

save_plot(clusters_all_proj_plot, Path(outputs_folder, f"clusters_all_{projection_method.lower()}_plot.png"), dpi=150)